<a href="https://colab.research.google.com/github/CyberWarSmith/AXIOM/blob/main/Experiment_Colab_Notebook_SL_v0_1_(SOC_triage_abstain_and_request_loop).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install openai tenacity numpy pandas scipy tqdm

import os, json, hashlib, random, itertools, time, textwrap, re
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import userdata

SEED = 20260804
random.seed(SEED); np.random.seed(SEED)

HF_TOKEN_SECRET = 'HuggingFaceNew'
API_KEY = userdata.get(HF_TOKEN_SECRET)
BASE_URL = 'https://api.deepinfra.com/v1/openai'

RUN_ID   = 'SL_v0_1'
PILOT    = True          # Cell 8 enforces the reduced grid while True
OUT      = Path(f'/content/{RUN_ID}'); OUT.mkdir(exist_ok=True)
CKPT     = OUT / 'checkpoints'; CKPT.mkdir(exist_ok=True)

RUNS_PER_CELL   = 3
MAX_LOOP_ITERS  = 3       # tier 0 -> 1 -> 2, then forced terminal
TOKEN_BUDGET    = 3000    # total per trial across all loop iterations
TAU             = 0.60    # pre-registered point threshold
BOOTSTRAP_N     = 5000

GATES = {
    'blinding_max_recall'   : 0.45,
    'irr_min_kappa'         : 0.40,
    'fabricated_field_max'  : 0.05,
    'negative_control_max'  : 1.0,
}

print('config loaded', RUN_ID, 'PILOT' if PILOT else 'FULL')

In [ ]:
MODELS = {
    'qwen'  : 'Qwen/Qwen3-235B-A22B-Instruct-2507',
    'glm'   : 'zai-org/GLM-4.6',
    'llama' : 'meta-llama/Llama-3.3-70B-Instruct',
}
JUDGE_MODEL = MODELS['glm']

# CrowdStrike exclusion is an employment constraint. No CrowdStrike-derived
# model or dataset in this run. Recorded here so it appears in the hash.
EXCLUDED_VENDORS = ['crowdstrike']

PROVIDER_MAP = {m: 'deepinfra' for m in MODELS.values()}

def assert_providers():
    bad = [m for m, p in PROVIDER_MAP.items() if p != 'deepinfra']
    if bad:
        raise SystemExit(f'provider mismatch, refusing to run: {bad}')
    if JUDGE_MODEL in [MODELS['qwen']]:
        raise SystemExit('judge must not be a generator model in this run')
assert_providers()
print('providers locked')

In [ ]:
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
PERMANENT_CODES = {400, 401, 403, 404, 422}

class PermanentError(Exception): pass

@retry(stop=stop_after_attempt(5),
       wait=wait_exponential(min=2, max=60),
       retry=retry_if_exception_type(Exception),
       reraise=True)
def _call(model, messages, max_tokens, temperature):
    try:
        r = client.chat.completions.create(
            model=model, messages=messages,
            max_tokens=max_tokens, temperature=temperature,
            extra_body={'chat_template_kwargs': {'enable_thinking': False}},
        )
    except Exception as e:
        code = getattr(getattr(e, 'response', None), 'status_code', None)
        if code in PERMANENT_CODES:
            raise PermanentError(f'{code}: {e}') from e
        raise
    return r

def run_one(model, system, user, max_tokens=1200, temperature=0.7):
    """Returns (text, usage_dict). Token usage is recorded on every call
    because method rule 6 requires length alongside every quality metric."""
    msgs = [{'role': 'system', 'content': system},
            {'role': 'user',   'content': user}]
    r = _call(model, msgs, max_tokens, temperature)
    u = r.usage
    return r.choices[0].message.content, {
        'prompt_tokens': u.prompt_tokens,
        'completion_tokens': u.completion_tokens,
        'total_tokens': u.total_tokens,
    }

In [ ]:
def preflight():
    rows = []
    for name, m in MODELS.items():
        t, u = run_one(m, 'Answer in one word.', 'Say OK.', max_tokens=20, temperature=0.0)
        leaked = bool(re.search(r'<think>|</think>|^Thinking', t.strip(), re.I))
        rows.append({'model': name, 'reply': t.strip()[:40],
                     'thinking_leak': leaked, 'tokens': u['total_tokens']})
    df = pd.DataFrame(rows)
    if df.thinking_leak.any():
        raise SystemExit('thinking suppression failed, fix before generating')
    return df

preflight()